In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\tmp\\course-restructure-20261118\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0

# Разбор A. Оптимизация и валидация HPLC

Как выбрать рабочие условия, если таблица содержит несколько факторов и повторы в центре плана? На A учимся оценивать модель отклика, сохраняя B как отдельную опубликованную систему. Гипотеза: качество аппроксимации на обучении завышает пригодность для нового режима. Нужно проверять остатки и ошибки вне обучающей строки.

Постройте матрицу 20×10 по опубликованной таблице, оцените удерживание лактона. Проверьте ранг, RMSE на обучении и LOOCV. Затем исключайте целый уникальный режим факторов и сравните ошибку. Не используйте прогнозы для выдуманного «оптимума», если нет критериев всех аналитических ограничений.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M10-L02','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
import numpy as np,csv
with (DATA/'atorvastatin_hplc_table1.csv').open(encoding='utf-8',newline='') as f:rows=list(csv.DictReader(f))
factors=np.array([[float(r[k]) for k in ['acetonitrile_percent','column_temperature_C','flow_mL_min']] for r in rows])
x=(factors-[65,35,1])/[5,5,.2]
X=np.column_stack([np.ones(len(x)),x,x*x,x[:,0]*x[:,1],x[:,0]*x[:,2],x[:,1]*x[:,2]])
y=np.array([float(r['retention_ATO_lactone_min']) for r in rows]);assert X.shape==(20,10)
beta,_,rank,_=np.linalg.lstsq(X,y,rcond=None);assert rank==10
pred=X@beta;train=float(np.sqrt(np.mean((pred-y)**2)));print('Train RMSE / min',train)


Train RMSE / min 0.16726989078622703


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
loo=np.zeros(len(y));group=np.zeros(len(y))
for i in range(len(y)):
    keep=np.arange(len(y))!=i;loo[i]=X[i]@np.linalg.lstsq(X[keep],y[keep],rcond=None)[0]
for condition in np.unique(factors,axis=0):
    test=np.all(factors==condition,axis=1);group[test]=X[test]@np.linalg.lstsq(X[~test],y[~test],rcond=None)[0]
print('LOOCV, grouped RMSE / min',np.sqrt(np.mean((loo-y)**2)),np.sqrt(np.mean((group-y)**2)))
fig,ax=plt.subplots();ax.scatter(y,group-y);ax.axhline(0,color='gray');ax.set(xlabel='Measured retention / min',ylabel='Held-condition residual / min');publish_plot(fig,'hplc_cv.png')


LOOCV, grouped RMSE / min 0.24636719045966815 0.232363784759675


<Figure size 640x480 with 1 Axes>

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
result={'kind':'calculated','rank':int(rank),'rows':len(y),'train_RMSE_min':train,
        'LOOCV_RMSE_min':float(np.sqrt(np.mean((loo-y)**2))),'condition_RMSE_min':float(np.sqrt(np.mean((group-y)**2)))}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M10-L02','A',result,ROOT,inputs=['practicuum/shared/data/atorvastatin_hplc_table1.csv'])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\course-restructure-20261118\practicuum\evidence\20261118-full-course-windows\workspace\my_research\evidence\M10-L02\20261118T141625359734Z-A-f52f20cd452d4d809b0eed3c56b1775c.json


Оптимизация требует селективности, runtime, pressure, устойчивости и проверки выбранного режима. Низкое RMSE одной колонки не завершает валидацию аналитического метода. После аналитики вернёмся к биологической мишени и уровням доказательств.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?